In [ ]:
# Optional local install if you need extra provider SDKs:
# pip install together


## Reproduction Notes
This notebook reproduces the WorldValue quantile analyses and figure outputs used in the paper.

- Retained question ids come from `data/worldvalue/retained_questions_235.json`.
- Archived PNG copies of the final paper plots are stored in `figures/`.
- The main plotting blocks below are annotated with the paper figure numbers they generate.


In [ ]:
# NOTE: Colab-only setup disabled for local environment.
# Keep this block for reference if you later run in Colab.
# import sys
# import os
# from google.colab import userdata, drive
# drive.mount('/content/drive')

# os.chdir('/content/drive/MyDrive/simulation_code/')

# sys.path.insert(0, '/content/drive/MyDrive/simulation_code') # Replace with your actual folder path

### Repository Root Setup
The notebook locates the repository root automatically so it can be run from this folder or from the repo root.


In [ ]:
import os, sys
from pathlib import Path

NOTEBOOK_DIR = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    direct = candidate / 'wvs_notebook_helpers.py'
    nested = candidate / 'paper_reproduction' / 'worldvalue_quantile' / 'wvs_notebook_helpers.py'
    if direct.exists():
        NOTEBOOK_DIR = candidate
        break
    if nested.exists():
        NOTEBOOK_DIR = nested.parent
        break
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('Could not locate paper_reproduction/worldvalue_quantile.')

from wvs_notebook_helpers import find_repo_root, worldvalue_figures_dir

ROOT = find_repo_root(NOTEBOOK_DIR)
FIGURES_DIR = worldvalue_figures_dir(ROOT)
os.chdir(ROOT)
for entry in [NOTEBOOK_DIR, ROOT]:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))
print('Using reproduction root:', ROOT)
print('Using notebook directory:', NOTEBOOK_DIR)
print('Using figure archive:', FIGURES_DIR)


In [ ]:
# Add the current directory to the Python path to import local modules
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

In [ ]:

import json, pickle, re, gc, asyncio, random, time, glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from typing import Optional, List, Dict
#from together import Together
#from together import AsyncTogether  # built-in async client
from collections import deque
from difflib import SequenceMatcher
from collections import OrderedDict, defaultdict
from scipy.stats import linregress, t
from sklearn.metrics import mean_squared_error
from matplotlib.cm import get_cmap
from matplotlib.lines import Line2D
from math import sqrt, isfinite
from simfidelity_utils import *
from simfidelity_utils import _as_dict_series, _dropna_np
from numpy.random import default_rng

from wvs_notebook_helpers import filter_mapping_to_questions, install_numpy_pickle_compat, load_retained_questions

install_numpy_pickle_compat()


In [ ]:
# File loading (retained 235-question experiment subset)
retained_questions = load_retained_questions()

gpt4o_file = 'data/worldvalue/synthetic answers/clean/synthetic_answers_clean_gpt4o.pkl'
with open(gpt4o_file, 'rb') as f:
    gpt4o = filter_mapping_to_questions(pickle.load(f), retained_questions)

gpt5m_file = 'data/worldvalue/synthetic answers/clean/synthetic_answers_clean_gpt-5-mini.pkl'
with open(gpt5m_file, 'rb') as f:
    gpt5m = filter_mapping_to_questions(pickle.load(f), retained_questions)

llama_file = 'data/worldvalue/synthetic answers/clean/synthetic_answers_clean_llama-3.3.pkl'
with open(llama_file, 'rb') as f:
    llama = filter_mapping_to_questions(pickle.load(f), retained_questions)

qwen_file = 'data/worldvalue/synthetic answers/clean/synthetic_answers_clean_Qwen3-235B.pkl'
with open(qwen_file, 'rb') as f:
    qwen = filter_mapping_to_questions(pickle.load(f), retained_questions)

uni_file = 'data/worldvalue/synthetic answers/clean/uniform_benchmark.pkl'
with open(uni_file, 'rb') as f:
    uniform_benchmark = filter_mapping_to_questions(pickle.load(f), retained_questions)

full_file = 'data/worldvalue/full_population_response_clean.pkl'
with open(full_file, 'rb') as f:
    full = filter_mapping_to_questions(pickle.load(f), retained_questions)

print(f'Retained questions loaded: {len(retained_questions)}')
print({
    'full': len(full),
    'gpt4o': len(gpt4o),
    'gpt5mini': len(gpt5m),
    'llama': len(llama),
    'qwen3': len(qwen),
    'uniform': len(uniform_benchmark),
})


## Calibrated Quantile Curve Construction

## Archived Paper Figures
Archived PNG versions of the WorldValue paper figures are stored in `figures/`.

### Figure 3 — Calibrated quantile curves
![Figure 3](figures/worldvalue_calibrated_quantile_curve.png)

### Figure 4 — Robustness across human sample sizes
![Figure 4](figures/worldvalue_robustness_all_models_n_grid.png)

### Figure 5 — Tightness with adaptive $\gamma_j$
![Figure 5](figures/worldvalue_tightness_adaptive_gamma_gpt4o.png)

### Figure 6 — Tightness with fixed $\gamma$
![Figure 6](figures/worldvalue_tightness_fixed_gamma_gpt4o.png)

### Figure 7 — Confidence bands for GPT-4o vs Llama 3.3
![Figure 7](figures/worldvalue_band_panels_gpt4o_vs_llama_n1000_n5000.png)

For the full archive and supplementary plot notes, see `figures/README.md`.

### Supplementary Figure — Beta sensitivity for $\gamma_j = 1 - n_j^{-\beta}$
![Supplementary beta sensitivity](figures/worldvalue_beta_sensitivity_gpt4o_n50_n200.png)


In [ ]:
# ============================================================
# (A) Human-side subsampling (controls n_j; handles missingness)
# ============================================================

def _coerce_1d_float_array(x):
    """
    Convert array-like / pd.Series to 1D float np.ndarray (keeps NaNs for now).

    Note:
    - This will raise if x contains non-numeric strings. If that is a concern,
      replace with pd.to_numeric(..., errors="coerce").
    """
    if hasattr(x, "values"):
        x = x.values
    arr = np.asarray(x, dtype=float).ravel()
    return arr

def _subsample_human(yh, n_target=None, rng=None, replace=False):
    """
    Subsample human responses for one question.

    - Drops NaNs/infs first via np.isfinite.
    - If n_target is None: returns all non-missing.
    - Else: samples min(n_target, n_available) without replacement by default.

    Returns
    -------
    y_sub : np.ndarray (float), finite-only
    n_eff : int  (effective sample size used)
    """
    if rng is None:
        rng = np.random.default_rng()

    y = _coerce_1d_float_array(yh)
    y = y[np.isfinite(y)]
    n_avail = int(y.size)

    if n_avail == 0:
        return np.asarray([], dtype=float), 0

    if (n_target is None) or (int(n_target) <= 0):
        return y, n_avail

    n_use = int(min(int(n_target), n_avail))

    if (not replace) and (n_use == n_avail):
        return y, n_avail

    idx = rng.choice(n_avail, size=n_use, replace=bool(replace))
    return y[idx], n_use


# ============================================================
# (B) Gamma schedule g(n): separated and swappable
#     DEFAULT: adaptive gamma_j = 1 - 1/n^(1/3)
# ============================================================

def gamma_schedule_power_1_3(n_eff: int, eps: float = 1e-6) -> float:
    """
    Default adaptive schedule:
        g(n) = 1 - 1 / n^(1/3),
    clipped into (eps, 1-eps).

    - For n_eff <= 0: returns NaN (caller should skip).
    """
    n_eff = int(n_eff)
    if n_eff <= 0:
        return np.nan
    g = 1.0 - (n_eff ** (-1.0 / 3.0))
    return float(np.clip(g, eps, 1.0 - eps))


def _resolve_gamma_j(
    n_eff: int,
    gamma_fixed: float,
    use_adaptive_gamma: bool,
    gamma_fn,
    gamma_fn_kwargs: dict,
) -> float:
    """
    Decide per-scenario gamma_j.

    - Default: adaptive gamma_j = gamma_fn(n_eff, **gamma_fn_kwargs)
    - If use_adaptive_gamma=False: gamma_j = gamma_fixed
    """
    if not use_adaptive_gamma:
        return float(gamma_fixed)
    return float(gamma_fn(int(n_eff), **(gamma_fn_kwargs or {})))


# ============================================================
# (C) Batch Δ̂^+ for many simulators with optional adaptive gamma_j
#     DEFAULT: adaptive gamma_j
# ============================================================

def compute_deltas_multi(
    actual_dict,
    simulators_dict,
    k=500,
    n_target=None,
    gamma=0.5,
    use_adaptive_gamma=True,
    gamma_fn=gamma_schedule_power_1_3,
    gamma_fn_kwargs=None,
    ci_family="bounded",
    loss_kind="abs",
    ci_kwargs=None,
    seed=0,
    human_replace=False,
    return_df=False,
    also_return_dict=False,
):
    """
    Compute per-scenario upper pseudo-discrepancies Δ-hat^+ for multiple simulators.

    Human subsamples and adaptive gamma_j values are now
    precomputed once per qid and then shared across simulators, so cross-model
    comparisons use the same human-side uncertainty set.
    """
    ci_kwargs = ci_kwargs or {}
    gamma_fn_kwargs = gamma_fn_kwargs or {}

    common = set(actual_dict.keys())
    for sim in simulators_dict.values():
        common &= set(sim.keys())
    common = sorted(common)

    rng_master = np.random.default_rng(seed)
    out = {}
    rows = []
    gamma_mode = "adaptive" if use_adaptive_gamma else "fixed"

    human_sub = {}
    n_eff_by_qid = {}
    gamma_by_qid = {}
    seed_by_qid = {}

    for qid in common:
        local_seed = int(rng_master.integers(0, 2**32 - 1))
        rng_local = np.random.default_rng(local_seed)
        yh_sub, n_eff = _subsample_human(
            actual_dict[qid], n_target=n_target, rng=rng_local, replace=human_replace
        )
        if n_eff <= 0:
            continue

        gamma_j = _resolve_gamma_j(
            n_eff=n_eff,
            gamma_fixed=float(gamma),
            use_adaptive_gamma=bool(use_adaptive_gamma),
            gamma_fn=gamma_fn,
            gamma_fn_kwargs=gamma_fn_kwargs,
        )
        if (not np.isfinite(gamma_j)) or (gamma_j <= 0.0) or (gamma_j >= 1.0):
            continue

        human_sub[qid] = yh_sub
        n_eff_by_qid[qid] = int(n_eff)
        gamma_by_qid[qid] = float(gamma_j)
        seed_by_qid[qid] = int(local_seed)

    used_qids = sorted(human_sub.keys())

    for sim_idx, (sim_name, sim) in enumerate(simulators_dict.items()):
        ds = []
        for qid in used_qids:
            ym = sim[qid]
            if hasattr(ym, "values"):
                ym = ym.values

            rng_local = np.random.default_rng(seed_by_qid[qid] + 1009 * (sim_idx + 1))
            d, info = compute_pseudo_delta(
                y_human=human_sub[qid],
                y_model=ym,
                k=int(k),
                gamma=float(gamma_by_qid[qid]),
                ci_family=ci_family,
                loss_kind=loss_kind,
                ci_kwargs=ci_kwargs,
                rng=rng_local,
            )

            if not np.isfinite(d):
                continue

            ds.append((qid, float(d)))
            rows.append({
                "qid": qid,
                "sim": sim_name,
                "delta": float(d),
                "n_eff": int(n_eff_by_qid[qid]),
                "k": int(k),
                "gamma_j": float(gamma_by_qid[qid]),
                "gamma_mode": gamma_mode,
            })

        out[sim_name] = dict(ds)

    if rows:
        df_tmp = pd.DataFrame(rows)
        bar_gamma_overall = float(df_tmp["gamma_j"].mean())
        g_by_sim = df_tmp.groupby("sim", sort=True)["gamma_j"].mean()
        n_by_sim = df_tmp.groupby("sim", sort=True)["gamma_j"].size()
        gamma_summary = {
            "gamma_mode": gamma_mode,
            "bar_gamma_overall": bar_gamma_overall,
            "bar_gamma_by_sim": {str(k): float(v) for k, v in g_by_sim.items()},
            "n_by_sim": {str(k): int(v) for k, v in n_by_sim.items()},
        }
    else:
        df_tmp = pd.DataFrame(columns=["qid", "sim", "delta", "n_eff", "k", "gamma_j", "gamma_mode"])
        gamma_summary = {
            "gamma_mode": gamma_mode,
            "bar_gamma_overall": np.nan,
            "bar_gamma_by_sim": {},
            "n_by_sim": {},
        }

    if return_df:
        if also_return_dict:
            return out, used_qids, df_tmp, gamma_summary
        return df_tmp, used_qids, gamma_summary

    return out, used_qids, gamma_summary


def plot_upper_gamma_indexed_multi_adaptive(
    deltas_data,
    gamma_summary=None,          # from compute_deltas_multi (recommended)
    bar_gamma=None,              # optional override
    sim_names=None,
    tau_grid=None,
    color_map=None,
    linewidth=2.5,
    linestyle="-",
    legend=True,
    xlabel=r"$\tau$",
    ylabel=None,
    title=None,
    figsize=(7.5, 5),
):
    """
    Correct asymptotic index map (your Eq. (asmptotic_guarantee)):
        tau ↦ V̂_m( (1 - bar_gamma) + bar_gamma * tau )
    """
    if isinstance(deltas_data, dict):
        deltas_dict = {}
        for k, v in deltas_data.items():
            if isinstance(v, dict):
                arr = np.asarray(list(v.values()), float)
            else:
                arr = np.asarray(v, float)
            deltas_dict[k] = arr
    else:
        deltas_dict = _as_deltas_dict(deltas_data)

    if bar_gamma is None:
        if gamma_summary is None:
            raise ValueError("Provide bar_gamma or gamma_summary (returned by compute_deltas_multi).")
        bar_gamma = float(gamma_summary.get("bar_gamma_overall", np.nan))

    if (not np.isfinite(bar_gamma)) or (bar_gamma <= 0.0) or (bar_gamma >= 1.0):
        raise ValueError(f"Invalid bar_gamma={bar_gamma}. Expected in (0,1).")

    if sim_names is None:
        sim_names = list(deltas_dict.keys())
    if tau_grid is None:
        tau_grid = np.linspace(0.0, 1.0, 201)

    tau_grid = np.asarray(tau_grid, float)

    # ---- FIX: alpha_shifted = (1 - bar_gamma) + bar_gamma * tau
    alpha_shifted = np.clip((1.0 - bar_gamma) + bar_gamma * tau_grid, 0.0, 1.0)

    fig, ax = plt.subplots(figsize=figsize)
    out = {}

    for name in sim_names:
        if name not in deltas_dict:
            continue

        deltas = np.asarray(deltas_dict[name], float)
        deltas = deltas[np.isfinite(deltas)]
        if deltas.size == 0:
            continue

        V_upper = empirical_quantile_curve(deltas, alpha_shifted)

        color = None if color_map is None else color_map.get(name, None)
        ax.plot(
            tau_grid,
            V_upper,
            linestyle=linestyle,
            linewidth=linewidth,
            color=color,
            label=str(name),
        )

        out[name] = {
            "tau": tau_grid,
            "alpha_shifted": alpha_shifted,
            "V_upper": V_upper,
            "bar_gamma": float(bar_gamma),
        }

    ax.set_xlabel(xlabel)
    if ylabel is None:
        ylabel = r"$\widehat V_m\!\left((1-\bar\gamma)+\bar\gamma\,\tau\right)$"
    ax.set_ylabel(ylabel)

    if title is None:
        title = rf"Upper curve with adaptive $\bar\gamma$ = {bar_gamma:.3f}"
    ax.set_title(title)

    if legend:
        ax.legend(fontsize=10)

    fig.tight_layout()
    plt.show()
    out["_figure"] = fig
    out["_axes"] = ax
    return out



### Paper Figure 3 — Calibrated Quantile Curve
This block reproduces the main calibrated quantile curve across simulators shown in Figure 3 of the paper.
Archived PNG: `figures/worldvalue_calibrated_quantile_curve.png`.


In [ ]:
# Your simulator bundle
simulators = {
    "GPT4o": gpt4o,
    "Llama": llama,
    "GPT5mini": gpt5m,
    "Qwen3": qwen,
    "Uniform": uniform_benchmark,
}

SIM_COLOR_MAP = {
    "GPT4o":   "C0",
    "Llama":   "C1",
    "GPT5mini":"C2",
    "Uniform": "C3",
    "Qwen3":   "C4",
}

CI_FAMILY = "bounded"
LOSS_KIND = "sq"
GAMMA = 0.50
K_MODEL = 200
N_HUMAN = 500
SEED = 0

CI_KWARGS = {
    "bounds": (-1.0, 1.0),
    "method": "hoeffding",
}

# ------------------------------------------------------------
# 1) Compute Δ̂^+ with adaptive gamma_j (DEFAULT in your new compute_deltas_multi)
# ------------------------------------------------------------
deltas_dict, common_qids, gamma_summary = compute_deltas_multi(
    actual_dict=full,
    simulators_dict=simulators,
    k=K_MODEL,
    n_target=N_HUMAN,
    gamma=GAMMA,                 # kept for compatibility; ignored in adaptive mode
    # use_adaptive_gamma=True,    # default; include if you want explicitness
    ci_family=CI_FAMILY,
    loss_kind=LOSS_KIND,
    ci_kwargs=CI_KWARGS,
    seed=SEED,
    human_replace=False,
    return_df=False,
)

print("bar_gamma_overall:", gamma_summary["bar_gamma_overall"])
print("bar_gamma_by_sim:", gamma_summary["bar_gamma_by_sim"])

calibrated_plot = plot_upper_gamma_indexed_multi_adaptive(
    deltas_data=deltas_dict,
    gamma_summary=gamma_summary,   # uses bar_gamma_overall
    sim_names=["GPT4o", "GPT5mini", "Llama", "Qwen3", "Uniform"],
    color_map=SIM_COLOR_MAP,
)
calibrated_fig_path = FIGURES_DIR / "worldvalue_calibrated_quantile_curve.png"
calibrated_plot["_figure"].savefig(calibrated_fig_path, dpi=200, bbox_inches="tight")
plt.close(calibrated_plot["_figure"])
print(f"Saved calibrated quantile figure to: {calibrated_fig_path}")


## LLM Performance Check - Different n fidelity holds

In [ ]:
def _gamma_power_1_3(n_eff: int, eps: float = 1e-6) -> float:
    """Default adaptive schedule: gamma_j = 1 - 1 / n_eff^(1/3), clipped to (eps, 1-eps)."""
    n_eff = int(n_eff)
    if n_eff <= 0:
        return np.nan
    g = 1.0 - (n_eff ** (-1.0 / 3.0))
    return float(np.clip(g, eps, 1.0 - eps))


def plot_empirical_quantiles_all_models_one_n(
    full_human, simulators, n, k,
    ci_family="bounded", loss_kind="sq",
    gamma=0.5,                   # kept for compatibility; used only if use_adaptive_gamma=False
    ci_kwargs=None, seed=0,
    ax=None, show=True, add_legend=True, title=None,
    index_mode="gamma",           # "gamma" or "empirical"
    tau_grid=None,               # defaults to np.linspace(0,1,201)
    name_map=None,               # {"gpt4o":"GPT4o", ...}
    color_map=None,              # {"GPT4o":"tab:blue", ...} (display-name keyed)

    # NEW: adaptive gamma_j controls
    use_adaptive_gamma=True,     # default: adaptive gamma_j = g(n_j)
    gamma_fn=_gamma_power_1_3,
    gamma_fn_kwargs=None,
    bar_gamma_mode="per_sim",    # "per_sim" (recommended) or "global"
):
    """
    Plot V̂_m curves for multiple simulators at a fixed human size n and model budget k.

    NEW (adaptive n_j / gamma_j):
      - For each qid, subsample humans to n_used (after dropping NaNs).
      - Set gamma_j = g(n_used) (default 1 - 1/n_used^(1/3)).
      - Use the SAME human subsample and gamma_j for all simulators (per qid),
        so simulator comparisons are on the same human-side CI.

    Curve indexing:
      - index_mode="gamma":
            uses the theorem’s asymptotic affine map with bar_gamma:
                tau -> V̂_m( (1 - bar_gamma) + bar_gamma * tau )
        where bar_gamma is either:
            * per-sim average over qids that produced finite deltas ("per_sim"), or
            * global average over qids with valid human subsample ("global").
      - index_mode="empirical":
            tau -> V̂_m(tau)
    """
    rng = default_rng(seed)
    ci_kwargs = ci_kwargs or {}
    gamma_fn_kwargs = gamma_fn_kwargs or {}

    if tau_grid is None:
        tau_grid = np.linspace(0.0, 1.0, 201)
    tau_grid = np.asarray(tau_grid, float)

    # intersect qids
    common = set(full_human.keys())
    for s in simulators.values():
        common &= set(s.keys())
    qids = sorted(common)
    if not qids:
        raise ValueError("No overlapping qids across human and simulators.")

    # ------------------------------------------------------------
    # (1) Precompute human subsamples ONCE per qid + gamma_j(qid)
    #     (shared across simulators)
    # ------------------------------------------------------------
    human_sub = {}     # qid -> np.ndarray
    gamma_by_qid = {}  # qid -> gamma_j
    n_used_by_qid = {} # qid -> n_used

    for q in qids:
        y_h_all = np.asarray(getattr(full_human[q], "values", full_human[q]), float)
        y_h_all = y_h_all[np.isfinite(y_h_all)]
        if y_h_all.size == 0:
            continue

        n_used = int(min(int(n), int(y_h_all.size)))
        idx_h = rng.choice(y_h_all.size, size=n_used, replace=False)
        y_h = y_h_all[idx_h]

        if n_used <= 0:
            continue

        if use_adaptive_gamma:
            gamma_j = float(gamma_fn(n_used, **gamma_fn_kwargs))
        else:
            gamma_j = float(gamma)

        if (not np.isfinite(gamma_j)) or (gamma_j <= 0.0) or (gamma_j >= 1.0):
            continue

        human_sub[q] = y_h
        gamma_by_qid[q] = gamma_j
        n_used_by_qid[q] = n_used

    qids_used = sorted(human_sub.keys())
    if not qids_used:
        raise ValueError("No qids left after human filtering/subsampling.")

    bar_gamma_global = float(np.mean([gamma_by_qid[q] for q in qids_used]))

    # ------------------------------------------------------------
    # (2) Compute Δ̂ per simulator (reusing human_sub[q] and gamma_j(q))
    # ------------------------------------------------------------
    deltas_by_sim = {}
    gamma_rows_by_sim = {}

    for key, simmap in simulators.items():
        ds = []
        g_rows = []
        for q in qids_used:
            y_h = human_sub[q]
            gamma_j = gamma_by_qid[q]

            y_m_all = simmap[q]
            val, _ = compute_pseudo_delta(
                y_human=y_h,
                y_model=y_m_all,      # do NOT subsample model here; utils handles k
                k=int(k),
                gamma=float(gamma_j), # per-qid gamma_j
                ci_family=ci_family,
                loss_kind=loss_kind,
                ci_kwargs=ci_kwargs,
                rng=rng,
            )

            if np.isfinite(val):
                ds.append(float(val))
                g_rows.append(float(gamma_j))

        deltas_by_sim[key] = np.asarray(ds, float)
        gamma_rows_by_sim[key] = np.asarray(g_rows, float)

    # ------------------------------------------------------------
    # (3) Axes setup (unchanged)
    # ------------------------------------------------------------
    created = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))
        created = True

    # ------------------------------------------------------------
    # (4) Plot curves (same layout; only index computation changes)
    # ------------------------------------------------------------
    for key, vals in deltas_by_sim.items():
        vals = np.asarray(vals, float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            continue

        disp = name_map.get(key, key) if name_map else key
        color = color_map.get(disp, None) if color_map else None

        # choose bar_gamma used for indexing
        if index_mode == "gamma":
            if bar_gamma_mode == "global":
                bar_gamma = bar_gamma_global
            elif bar_gamma_mode == "per_sim":
                gvec = gamma_rows_by_sim.get(key, None)
                if gvec is None or gvec.size == 0:
                    bar_gamma = bar_gamma_global
                else:
                    bar_gamma = float(np.mean(gvec))
            else:
                raise ValueError("bar_gamma_mode must be 'per_sim' or 'global'.")

            # IMPORTANT: correct asymptotic affine index map:
            #   alpha_shifted = (1 - bar_gamma) + bar_gamma * tau
            levels = np.clip((1.0 - bar_gamma) + bar_gamma * tau_grid, 0.0, 1.0)
            xvals = tau_grid

        elif index_mode == "empirical":
            levels = tau_grid
            xvals = tau_grid

        else:
            raise ValueError("index_mode must be 'gamma' or 'empirical'.")

        curve = empirical_quantile_curve(vals, levels)
        ax.plot(xvals, curve, lw=2, label=disp, color=color)

    # styling (kept intact)
    ax.set_xlim(0.0, 1.0)
    ax.set_ylabel("")  # keep empty like your previous
    ax.set_xlabel(r"$\tau$")
    if title:
        ax.set_title(title)
    if add_legend:
        ax.legend()

    if created and show:
        plt.tight_layout()
        plt.show()

    # optional return for diagnostics (non-breaking)
    return {
        "qids_used": qids_used,
        "bar_gamma_global": bar_gamma_global,
        "bar_gamma_by_sim": {k: (float(np.mean(v)) if v.size else np.nan) for k, v in gamma_rows_by_sim.items()},
        "n_used_by_qid": n_used_by_qid,
    }




### Paper Figure 4 — Robustness Across Human Sample Sizes
This block reproduces the four-panel robustness check over `n in {50, 500, 5000, 10000}` shown in Figure 4 of the paper.
This block saves directly to `figures/worldvalue_robustness_all_models_n_grid.png`.


In [ ]:
MODEL_ORDER = ["GPT4o", "Llama", "GPT5mini", "Qwen3", "Uniform"]
NAME_MAP = {"gpt4o":"GPT4o","llama":"Llama","gpt5-mini":"GPT5mini","qwen":"Qwen3","Uniform":"Uniform"}

rows, cols = 2, 2
n_values = [50, 500, 5000, 10000]

fig, axes = plt.subplots(rows, cols, figsize=(12.8, 8.0), sharex=True, sharey=True)
axes = axes.ravel()

for idx, (ax, n) in enumerate(zip(axes, n_values)):
    plot_empirical_quantiles_all_models_one_n(
        full_human=full,                 # <- FULL dataset
        simulators=simulators,
        n=n, k=200,
        ci_family="bounded", loss_kind="sq",
        gamma=0.5,                       # used only if use_adaptive_gamma=False
        ci_kwargs={"method": "hoeffding", "bounds": (-1.0, 1.0)},
        seed=42,
        title=f"All models @ n={n}, k=200",
        ax=ax, show=False, add_legend=False,
        index_mode="gamma",
        tau_grid=np.linspace(0, 1, 401),
        name_map=NAME_MAP, color_map=SIM_COLOR_MAP,

        # NEW: adaptive gamma_j enabled by default; included for explicitness
        use_adaptive_gamma=True,
        gamma_fn=_gamma_power_1_3,
        gamma_fn_kwargs={"eps": 1e-6},
        bar_gamma_mode="per_sim",   # or "global"
    )

    # force y ticks + labels on every subplot
    ax.set_ylim(0, 4)
    ax.set_yticks([0, 1, 2, 3, 4])
    ax.tick_params(axis="y", which="both", labelleft=True)

    # x-axis uses tau throughout: the displayed quantile coordinate
    if idx // cols == rows - 1:
        ax.set_xlabel(r"$\tau$")

# single shared legend with a fixed model order so all plotted models appear consistently
legend_labels = [name for name in MODEL_ORDER if name in SIM_COLOR_MAP]
legend_handles = [Line2D([0], [0], color=SIM_COLOR_MAP[name], lw=2.4) for name in legend_labels]
fig.subplots_adjust(right=0.84, bottom=0.10, wspace=0.16, hspace=0.22)
fig.legend(
    legend_handles,
    legend_labels,
    title="Model",
    loc="center left",
    bbox_to_anchor=(0.86, 0.5),
    ncol=1,
    frameon=False,
)
robustness_fig_path = FIGURES_DIR / "worldvalue_robustness_all_models_n_grid.png"
fig.savefig(robustness_fig_path, dpi=200, bbox_inches="tight")
print(f"Saved robustness figure to: {robustness_fig_path}")
plt.close(fig)


## Conservative Analysis - Different n Tightness

### Paper Figures 5 and 6 — Tightness Analysis
This block reproduces the tightness plots for GPT-4o under adaptive and fixed gamma choices, corresponding to Figures 5 and 6 in the paper.
Archived PNGs: `figures/worldvalue_tightness_adaptive_gamma_gpt4o.png` and `figures/worldvalue_tightness_fixed_gamma_gpt4o.png`.


In [ ]:
def _gamma_power_1_3(n_eff: int, eps: float = 1e-6) -> float:
    """Default adaptive schedule: gamma_j = 1 - 1/n_eff^(1/3), clipped to (eps, 1-eps)."""
    n_eff = int(n_eff)
    if n_eff <= 0:
        return np.nan
    g = 1.0 - (n_eff ** (-1.0 / 3.0))
    return float(np.clip(g, eps, 1.0 - eps))
def _true_loss_scalar(p_star: float, qhat: float, loss_kind: str) -> float:
    """
    Oracle (unobservable in practice unless you use full-data truth_override):
        L(p_star, qhat)

    Supports bounded/bernoulli scalar losses used in your pipeline.
    """
    if loss_kind == "abs":
        return abs(p_star - qhat)
    if loss_kind in ("sq", "l2"):
        d = p_star - qhat
        return d * d
    if loss_kind == "tv":  # Bernoulli TV
        return abs(p_star - qhat)
    if loss_kind == "kl":  # Bernoulli KL(p||q)
        p = float(np.clip(p_star, 1e-12, 1 - 1e-12))
        q = float(np.clip(qhat,   1e-12, 1 - 1e-12))
        return p * np.log(p / q) + (1 - p) * np.log((1 - p) / (1 - q))
    raise ValueError(f"Unsupported loss_kind='{loss_kind}' for scalar oracle loss.")


def plot_tightness_over_n_one_sim_two_graphs(
    full_human,
    sim_answers,
    n_list=(100, 200, 500, 1000),
    k=200,
    gamma_fixed=0.50,
    ci_family="bounded",
    loss_kind="sq",
    ci_kwargs=None,
    seed=42,
    tau_grid=None,
    truth_override=None,
    title_fixed=None,
    title_adaptive=None,
    ax_fixed=None,
    ax_adaptive=None,
    show=True,

    # adaptive gamma controls
    gamma_fn=_gamma_power_1_3,
    gamma_fn_kwargs=None,
    bar_gamma_scope="n_specific",   # "n_specific" (recommended) or "global"
):
    """
    Produce TWO separate plots (fixed-gamma vs adaptive-gamma) for ONE simulator.

    Fixed plot:
      - oracle dotted: τ -> Quantile_τ( L(p*, qhat) )
      - fixed curves:  τ -> V̂_m( (1-γ_fixed) + γ_fixed * τ )

    Adaptive plot:
      - oracle dotted: same oracle
      - adaptive curves:
            gamma_j = g(n_j),
            bar_gamma(n) = average gamma_j over qids used at that n,
            τ -> V̂_m( (1-bar_gamma(n)) + bar_gamma(n) * τ )

    Simulator-side qhat_j is fixed across n by subsampling k model outputs once per qid.
    """
    ci_kwargs = ci_kwargs or {}
    gamma_fn_kwargs = gamma_fn_kwargs or {}

    if tau_grid is None:
        tau_grid = np.linspace(0.0, 1.0, 201)
    tau_grid = np.asarray(tau_grid, float)

    rng = default_rng(seed)

    # Normalize inputs (dict-of-series)
    full_dict = _as_dict_series(full_human)
    sim_dict  = _as_dict_series(sim_answers)

    # Align qids
    qids = sorted(set(full_dict).intersection(sim_dict))
    if not qids:
        raise ValueError("No overlapping qids between full_human and sim_answers.")

    # 1) Fix simulator-side subsample once per qid -> y_model_k[q], qhat[q]
    y_model_k = {}
    qhat = {}

    # 2) Define oracle truth p_star[q]
    p_star = {}

    for q in qids:
        yh_full = _dropna_np(full_dict[q])
        ym_full = _dropna_np(sim_dict[q])

        if yh_full.size == 0 or ym_full.size == 0:
            continue

        # oracle p*
        if truth_override is not None and q in truth_override:
            p_star[q] = float(truth_override[q])
        else:
            p_star[q] = float(yh_full.mean())

        # fixed model subsample for qhat
        k_use = int(min(int(k), int(ym_full.size)))
        idx_m = rng.choice(ym_full.size, size=k_use, replace=False)
        yk = ym_full[idx_m]
        y_model_k[q] = yk
        qhat[q] = float(yk.mean())

    qids_use = [q for q in qids if (q in p_star and q in y_model_k and np.isfinite(qhat[q]))]
    if not qids_use:
        raise ValueError("No usable qids after dropping missing human/model data.")

    # Oracle curve: τ -> quantile_τ( L(p*, qhat) )
    deltas_true = np.asarray([_true_loss_scalar(p_star[q], qhat[q], loss_kind) for q in qids_use], float)
    V_true = empirical_quantile_curve(deltas_true, tau_grid)

    # FIXED index levels (correct asymptotic affine map)
    alpha_shifted_fixed = np.clip((1.0 - gamma_fixed) + gamma_fixed * tau_grid, 0.0, 1.0)

    # Axes creation (two separate)
    created_fixed = False
    if ax_fixed is None:
        fig_f, ax_fixed = plt.subplots(figsize=(7.8, 5.0))
        created_fixed = True

    created_adapt = False
    if ax_adaptive is None:
        fig_a, ax_adaptive = plt.subplots(figsize=(7.8, 5.0))
        created_adapt = True

    # Plot oracle on both
    ax_fixed.plot(tau_grid, V_true, "k:", lw=3, label="Δ* (oracle)")
    ax_adaptive.plot(tau_grid, V_true, "k:", lw=3, label="Δ* (oracle)")

    # Diagnostics
    diag = {
        "tau": tau_grid,
        "V_true": V_true,
        "qids": qids_use,
        "fixed": {},
        "adaptive": {},
    }

    # Per-n curves
    for n in n_list:
        rng_n = default_rng(int(seed) + 10_000 + int(n))

        d_vals_fixed = []
        d_vals_adapt = []
        gamma_vals_adapt = []

        for q in qids_use:
            yh_full = _dropna_np(full_dict[q])
            if yh_full.size == 0:
                continue

            n_use = int(min(int(n), int(yh_full.size)))
            idx_h = rng_n.choice(yh_full.size, size=n_use, replace=False)
            y_h = yh_full[idx_h]
            if y_h.size == 0:
                continue

            # ---- FIXED gamma Δ̂^+
            val_f, _ = compute_pseudo_delta(
                y_human=y_h,
                y_model=y_model_k[q],
                k=int(len(y_model_k[q])),
                gamma=float(gamma_fixed),
                ci_family=ci_family,
                loss_kind=loss_kind,
                ci_kwargs=ci_kwargs,
                rng=rng_n,
            )
            if np.isfinite(val_f):
                d_vals_fixed.append(float(val_f))

            # ---- ADAPTIVE gamma_j Δ̂^+
            gamma_j = float(gamma_fn(int(n_use), **gamma_fn_kwargs))
            if np.isfinite(gamma_j) and 0.0 < gamma_j < 1.0:
                val_a, _ = compute_pseudo_delta(
                    y_human=y_h,
                    y_model=y_model_k[q],
                    k=int(len(y_model_k[q])),
                    gamma=float(gamma_j),
                    ci_family=ci_family,
                    loss_kind=loss_kind,
                    ci_kwargs=ci_kwargs,
                    rng=rng_n,
                )
                if np.isfinite(val_a):
                    d_vals_adapt.append(float(val_a))
                    gamma_vals_adapt.append(float(gamma_j))

        # ---- draw FIXED curve on fixed axis
        d_vals_fixed = np.asarray(d_vals_fixed, float)
        if d_vals_fixed.size > 0:
            curve_fixed = empirical_quantile_curve(d_vals_fixed, alpha_shifted_fixed)
            ax_fixed.plot(tau_grid, curve_fixed, lw=3, label=fr"$n={int(n)}$")
            diag["fixed"][int(n)] = {
                "deltas": d_vals_fixed,
                "alpha_shifted": alpha_shifted_fixed,
                "curve": curve_fixed,
            }

        # ---- draw ADAPTIVE curve on adaptive axis
        d_vals_adapt = np.asarray(d_vals_adapt, float)
        gamma_vals_adapt = np.asarray(gamma_vals_adapt, float)

        if d_vals_adapt.size > 0:
            if bar_gamma_scope == "n_specific":
                bar_gamma = float(np.mean(gamma_vals_adapt)) if gamma_vals_adapt.size else np.nan
            elif bar_gamma_scope == "global":
                bar_gamma = float(gamma_fn(int(n), **gamma_fn_kwargs))
            else:
                raise ValueError("bar_gamma_scope must be 'n_specific' or 'global'.")

            alpha_shifted_adapt = np.clip((1.0 - bar_gamma) + bar_gamma * tau_grid, 0.0, 1.0)
            curve_adapt = empirical_quantile_curve(d_vals_adapt, alpha_shifted_adapt)

            ax_adaptive.plot(tau_grid, curve_adapt, lw=3, label=fr"$n={int(n)}$  (γ={bar_gamma:.2f})")
            diag["adaptive"][int(n)] = {
                "deltas": d_vals_adapt,
                "bar_gamma": bar_gamma,
                "gamma_vals": gamma_vals_adapt,
                "alpha_shifted": alpha_shifted_adapt,
                "curve": curve_adapt,
            }

    # Styling: FIXED
    ax_fixed.set_xlabel(r"$\tau$")
    ax_fixed.set_ylabel(r"$\widehat V_m(\cdot)$")
    if title_fixed is not None:
        ax_fixed.set_title(title_fixed)
    else:
        ax_fixed.set_title(fr"Fixed γ = {gamma_fixed:.2f}")
    ax_fixed.legend(fontsize=11, ncol=2)
    ax_fixed.figure.tight_layout()

    # Styling: ADAPTIVE
    ax_adaptive.set_xlabel(r"$\tau$")
    ax_adaptive.set_ylabel(r"$\widehat V_m(\cdot)$")
    if title_adaptive is not None:
        ax_adaptive.set_title(title_adaptive)
    else:
        ax_adaptive.set_title(r"Adaptive γ(n) = 1 - n^{-1/3}")
    ax_adaptive.legend(fontsize=10, ncol=1)
    ax_adaptive.figure.tight_layout()

    if show and created_fixed:
        plt.show()
    if show and created_adapt:
        plt.show()

    diag["fig_fixed"] = ax_fixed.figure
    diag["fig_adaptive"] = ax_adaptive.figure
    return diag


diag = plot_tightness_over_n_one_sim_two_graphs(
    full_human=full,
    sim_answers=gpt4o,
    n_list=[50, 500, 1000, 5000],
    k=200,
    gamma_fixed=0.50,
    ci_family="bounded",
    loss_kind="sq",
    ci_kwargs={"method": "hoeffding", "bounds": (-1.0, 1.0)},
    seed=42,
    truth_override=None,
    title_fixed=None,
    title_adaptive=None,
    show=True,
    gamma_fn=_gamma_power_1_3,
    gamma_fn_kwargs={"eps": 1e-6},
    bar_gamma_scope="n_specific",
)
tightness_fixed_fig_path = FIGURES_DIR / "worldvalue_tightness_fixed_gamma_gpt4o.png"
tightness_adaptive_fig_path = FIGURES_DIR / "worldvalue_tightness_adaptive_gamma_gpt4o.png"
diag["fig_fixed"].savefig(tightness_fixed_fig_path, dpi=200, bbox_inches="tight")
diag["fig_adaptive"].savefig(tightness_adaptive_fig_path, dpi=200, bbox_inches="tight")
print(f"Saved fixed-gamma tightness figure to: {tightness_fixed_fig_path}")
print(f"Saved adaptive-gamma tightness figure to: {tightness_adaptive_fig_path}")


## Confidence Bands
The helper block below defines the adaptive band routines used to reproduce Paper Figure 7.


In [ ]:
# ============================================================
# (A) Gamma schedules (plug-in; keep separate so you can swap)
# ============================================================
def g_default_n13(n_used: int, clip=(1e-6, 1 - 1e-6)):
    """g(n) = 1 - 1 / n^(1/3), using n_used after NaN-drop + subsample."""
    n = max(1, int(n_used))
    g = 1.0 - (n ** (-1.0 / 3.0))
    return float(np.clip(g, clip[0], clip[1]))

def g_fixed(gamma: float):
    gamma = float(gamma)
    def _g(_n_used: int):
        return gamma
    return _g


# ============================================================
# (B) Compute Δ^+ and Δ^- with FIXED gammas (upper+lower)
#     - identical y_h / y_m subsamples used for + and -
#     - returns df_pm + gamma_summary (includes bar gammas)
# ============================================================
def compute_plus_minus_df_fixed_gamma(
    full_human,
    simulators_dict,
    *,
    n=None,
    k=200,
    gamma_u=0.50,
    gamma_l=0.90,
    ci_family="bounded",
    loss_kind="sq",
    ci_kwargs=None,
    seed=0,
):
    ci_kwargs = ci_kwargs or {}

    full_dict = _as_dict_series(full_human)

    common = set(full_dict.keys())
    sims_norm = {}
    for name, sim_map in simulators_dict.items():
        sim_map_norm = _as_dict_series(sim_map)
        sims_norm[name] = sim_map_norm
        common &= set(sim_map_norm.keys())
    common = sorted(common)
    if not common:
        raise ValueError("No overlapping qids across full_human and simulators.")

    rng = default_rng(seed)
    rows = []

    # one fixed gamma per qid (same across all)
    gamma_u = float(gamma_u)
    gamma_l = float(gamma_l)

    for sim_name, sim_map in sims_norm.items():
        for q in common:
            yh_all = _dropna_np(full_dict[q])
            ym_all = _dropna_np(sim_map[q])
            if yh_all.size == 0 or ym_all.size == 0:
                continue

            # human subsample
            if n is None:
                yh = yh_all
                n_used = int(yh.size)
            else:
                n_used = int(min(int(n), int(yh_all.size)))
                idx_h = rng.choice(yh_all.size, size=n_used, replace=False)
                yh = yh_all[idx_h]

            # model subsample (fixed for + and -)
            k_used = int(min(int(k), int(ym_all.size)))
            idx_m = rng.choice(ym_all.size, size=k_used, replace=False)
            ym = ym_all[idx_m]

            d_plus, _ = compute_pseudo_delta(
                y_human=yh,
                y_model=ym,
                k=int(len(ym)),
                gamma=gamma_u,
                ci_family=ci_family,
                loss_kind=loss_kind,
                ci_kwargs=ci_kwargs,
                rng=rng,
            )

            d_minus, _ = compute_pseudo_delta_lower(
                y_human=yh,
                y_model=ym,
                k=int(len(ym)),
                gamma=gamma_l,
                ci_family=ci_family,
                loss_kind=loss_kind,
                ci_kwargs=ci_kwargs,
                rng=rng,
            )

            if np.isfinite(d_plus) and np.isfinite(d_minus):
                rows.append({
                    "qid": q,
                    "sim": sim_name,
                    "delta_plus": float(d_plus),
                    "delta_minus": float(d_minus),
                    "n_used": int(n_used),
                    "k_used": int(k_used),
                    "gamma_u": float(gamma_u),
                    "gamma_l": float(gamma_l),
                })

    df = pd.DataFrame(rows, columns=["qid","sim","delta_plus","delta_minus","n_used","k_used","gamma_u","gamma_l"])
    gamma_summary = {
        "mode": "fixed",
        "bar_gamma_u": float(gamma_u),
        "bar_gamma_l": float(gamma_l),
        "n_qids": int(df["qid"].nunique()) if not df.empty else 0,
        "n_rows": int(df.shape[0]),
    }
    return df, gamma_summary


# ============================================================
# (C) Compute Δ^+ and Δ^- with ADAPTIVE gammas (upper+lower)
#     - gamma_u(q) = g_u(n_used(q)), gamma_l(q) = g_l(n_used(q))
#     - same y_h / y_m subsamples used for + and - per (qid, sim)
#     - returns df_pm + gamma_summary (bar gammas over qids, not rows)
# ============================================================
def compute_plus_minus_df_adaptive_gamma(
    full_human,
    simulators_dict,
    *,
    n=None,
    k=200,
    g_u=None,                 # callable: n_used -> gamma_u
    g_l=None,                 # callable: n_used -> gamma_l
    ci_family="bounded",
    loss_kind="sq",
    ci_kwargs=None,
    seed=0,
):
    ci_kwargs = ci_kwargs or {}
    g_u = g_default_n13 if g_u is None else g_u
    g_l = g_default_n13 if g_l is None else g_l

    full_dict = _as_dict_series(full_human)

    common = set(full_dict.keys())
    sims_norm = {}
    for name, sim_map in simulators_dict.items():
        sim_map_norm = _as_dict_series(sim_map)
        sims_norm[name] = sim_map_norm
        common &= set(sim_map_norm.keys())
    common = sorted(common)
    if not common:
        raise ValueError("No overlapping qids across full_human and simulators.")

    rng = default_rng(seed)
    rows = []

    # --- precompute (per-qid) human subsample + adaptive gammas so they are shared across simulators ---
    qid_to_yh = {}
    qid_to_n_used = {}
    qid_to_gu = {}
    qid_to_gl = {}

    for q in common:
        yh_all = _dropna_np(full_dict[q])
        if yh_all.size == 0:
            continue

        if n is None:
            yh = yh_all
            n_used = int(yh.size)
        else:
            n_used = int(min(int(n), int(yh_all.size)))
            idx_h = rng.choice(yh_all.size, size=n_used, replace=False)
            yh = yh_all[idx_h]

        gu = float(np.clip(g_u(n_used), 1e-6, 1 - 1e-6))
        gl = float(np.clip(g_l(n_used), 1e-6, 1 - 1e-6))

        qid_to_yh[q] = yh
        qid_to_n_used[q] = int(n_used)
        qid_to_gu[q] = gu
        qid_to_gl[q] = gl

    qids_use = sorted(qid_to_yh.keys())
    if not qids_use:
        raise ValueError("No usable qids after human NaN-drop/subsample.")

    # --- compute + / - per simulator ---
    for sim_name, sim_map in sims_norm.items():
        for q in qids_use:
            ym_all = _dropna_np(sim_map[q])
            if ym_all.size == 0:
                continue

            yh = qid_to_yh[q]
            n_used = qid_to_n_used[q]
            gu = qid_to_gu[q]
            gl = qid_to_gl[q]

            # model subsample (fixed for + and - for this (qid, sim))
            k_used = int(min(int(k), int(ym_all.size)))
            idx_m = rng.choice(ym_all.size, size=k_used, replace=False)
            ym = ym_all[idx_m]

            d_plus, _ = compute_pseudo_delta(
                y_human=yh,
                y_model=ym,
                k=int(len(ym)),
                gamma=gu,
                ci_family=ci_family,
                loss_kind=loss_kind,
                ci_kwargs=ci_kwargs,
                rng=rng,
            )

            d_minus, _ = compute_pseudo_delta_lower(
                y_human=yh,
                y_model=ym,
                k=int(len(ym)),
                gamma=gl,
                ci_family=ci_family,
                loss_kind=loss_kind,
                ci_kwargs=ci_kwargs,
                rng=rng,
            )

            if np.isfinite(d_plus) and np.isfinite(d_minus):
                rows.append({
                    "qid": q,
                    "sim": sim_name,
                    "delta_plus": float(d_plus),
                    "delta_minus": float(d_minus),
                    "n_used": int(n_used),
                    "k_used": int(k_used),
                    "gamma_u": float(gu),
                    "gamma_l": float(gl),
                })

    df = pd.DataFrame(rows, columns=["qid","sim","delta_plus","delta_minus","n_used","k_used","gamma_u","gamma_l"])

    # bar gammas over qids (not multiplied by #sims)
    gu_bar = float(np.mean([qid_to_gu[q] for q in qids_use]))
    gl_bar = float(np.mean([qid_to_gl[q] for q in qids_use]))

    gamma_summary = {
        "mode": "adaptive",
        "bar_gamma_u": gu_bar,
        "bar_gamma_l": gl_bar,
        "n_qids": int(len(qids_use)),
        "n_rows": int(df.shape[0]),
    }
    return df, gamma_summary


# ============================================================
# (D) Plot band for a simulator using either FIXED or ADAPTIVE indexing
#     - FIXED: uses provided gamma_u, gamma_l
#     - ADAPTIVE: uses bar_gamma_u/bar_gamma_l from df (consistent with your asymptotic form)
# ============================================================
def plot_quantile_band_fixed_or_adaptive(
    df_plus_minus,
    *,
    sim_name,
    mode,                 # "fixed" or "adaptive"
    gamma_u=None,
    gamma_l=None,
    tau_grid=None,
    color_map=None,
    figsize=(7.5, 5.0),
    title=None,
):
    sub = df_plus_minus[df_plus_minus["sim"] == sim_name]
    if sub.empty:
        raise ValueError(f"No rows for sim='{sim_name}'.")

    x_plus  = sub["delta_plus"].to_numpy(float)
    x_minus = sub["delta_minus"].to_numpy(float)

    if tau_grid is None:
        m_eff = max(1, min(x_plus.size, x_minus.size))
        eps = 1.0 / (m_eff + 1.0)
        tau = np.linspace(eps, 1.0 - eps, 235)
    else:
        tau = np.asarray(tau_grid, float)
        tau = np.clip(tau, 1e-6, 1.0 - 1e-6)

    mode = str(mode).lower()
    if mode == "fixed":
        if gamma_u is None or gamma_l is None:
            raise ValueError("mode='fixed' requires gamma_u and gamma_l.")
        gu = float(gamma_u)
        gl = float(gamma_l)
    elif mode == "adaptive":
        # asymptotic indexing uses bar gammas
        gu = float(sub["gamma_u"].mean())
        gl = float(sub["gamma_l"].mean())
    else:
        raise ValueError("mode must be 'fixed' or 'adaptive'.")

    alpha_low = np.clip(gl * tau, 0.0, 1.0)
    alpha_up  = np.clip(gu * tau + (1.0 - gu), 0.0, 1.0)

    Vminus = empirical_quantile_curve(x_minus, alpha_low)
    Vplus  = empirical_quantile_curve(x_plus,  alpha_up)

    color = None
    if color_map is not None:
        color = color_map.get(sim_name, None)

    fig, ax = plt.subplots(figsize=figsize)
    ax.fill_between(tau, Vminus, Vplus, alpha=0.15, color="grey")
    ax.plot(tau, Vminus, "--", lw=1.8, color=color, label=rf"Lower: $\hat V_m^-$")
    ax.plot(tau, Vplus,  "-.", lw=1.8, color=color, label=rf"Upper: $\hat V_m^+$")

    ax.set_xlabel(r"$\tau$")
    if title is not None:
        ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    plt.show()

    return {"tau": tau, "alpha_low": alpha_low, "alpha_up": alpha_up, "Vminus": Vminus, "Vplus": Vplus, "gu": gu, "gl": gl}

def plot_quantile_band_two_sims(
    df_plus_minus,
    *,
    sim_a,
    sim_b,
    mode="adaptive",          # "fixed" or "adaptive"
    gamma_u=None,             # required only if mode="fixed"
    gamma_l=None,             # required only if mode="fixed"
    tau_grid=None,
    color_map=None,           # dict: sim -> color
    colors=None,              # optional override: (color_a, color_b)
    figsize=(7.8, 5.2),
    title=None,
    show_legend=True,
    fill_alpha=0.12,
):
    """
    Plot quantile bands for TWO simulators on the same axes.

    Band construction matches your existing function:
      alpha_low = gl * tau
      alpha_up  = (1 - gu) + gu * tau
    where:
      - fixed mode: (gu, gl) = (gamma_u, gamma_l)
      - adaptive mode: (gu, gl) = mean of per-row gamma_u/gamma_l within each sim
        (i.e., using df columns; consistent with your current convention)
    """

    # ----- choose tau grid
    if tau_grid is None:
        # choose based on smallest effective sample among the two sims
        sub_a = df_plus_minus[df_plus_minus["sim"] == sim_a]
        sub_b = df_plus_minus[df_plus_minus["sim"] == sim_b]
        if sub_a.empty or sub_b.empty:
            missing = [s for s, sub in [(sim_a, sub_a), (sim_b, sub_b)] if sub.empty]
            raise ValueError(f"No rows for sim(s): {missing}")
        m_eff = max(1, min(sub_a.shape[0], sub_b.shape[0]))
        eps = 1.0 / (m_eff + 1.0)
        tau = np.linspace(eps, 1.0 - eps, 235)
    else:
        tau = np.asarray(tau_grid, float)
        tau = np.clip(tau, 1e-6, 1.0 - 1e-6)

    mode = str(mode).lower()
    if mode not in ("fixed", "adaptive"):
        raise ValueError("mode must be 'fixed' or 'adaptive'.")

    # ----- helper to compute band for one sim
    def _compute_band_for_sim(sim_name):
        sub = df_plus_minus[df_plus_minus["sim"] == sim_name]
        if sub.empty:
            raise ValueError(f"No rows for sim='{sim_name}'.")

        x_plus  = sub["delta_plus"].to_numpy(float)
        x_minus = sub["delta_minus"].to_numpy(float)

        if mode == "fixed":
            if gamma_u is None or gamma_l is None:
                raise ValueError("mode='fixed' requires gamma_u and gamma_l.")
            gu = float(gamma_u)
            gl = float(gamma_l)
        else:
            # your existing convention: use mean of gammas within this sim's rows
            gu = float(sub["gamma_u"].mean())
            gl = float(sub["gamma_l"].mean())

        alpha_low = np.clip(gl * tau, 0.0, 1.0)
        alpha_up  = np.clip((1.0 - gu) + gu * tau, 0.0, 1.0)

        Vminus = empirical_quantile_curve(x_minus, alpha_low)
        Vplus  = empirical_quantile_curve(x_plus,  alpha_up)

        return {
            "sub": sub,
            "gu": gu,
            "gl": gl,
            "alpha_low": alpha_low,
            "alpha_up": alpha_up,
            "Vminus": Vminus,
            "Vplus": Vplus,
        }

    out_a = _compute_band_for_sim(sim_a)
    out_b = _compute_band_for_sim(sim_b)

    # ----- colors
    if colors is not None:
        col_a, col_b = colors
    else:
        col_a = None if color_map is None else color_map.get(sim_a, None)
        col_b = None if color_map is None else color_map.get(sim_b, None)

    # ----- plot
    fig, ax = plt.subplots(figsize=figsize)

    # sim A
    ax.fill_between(tau, out_a["Vminus"], out_a["Vplus"], alpha=fill_alpha, color=col_a)
    ax.plot(tau, out_a["Vminus"], "--", lw=1.8, color=col_a, label=f"{sim_a} lower")
    ax.plot(tau, out_a["Vplus"],  "-.", lw=1.8, color=col_a, label=f"{sim_a} upper")

    # sim B
    ax.fill_between(tau, out_b["Vminus"], out_b["Vplus"], alpha=fill_alpha, color=col_b)
    ax.plot(tau, out_b["Vminus"], "--", lw=1.8, color=col_b, label=f"{sim_b} lower")
    ax.plot(tau, out_b["Vplus"],  "-.", lw=1.8, color=col_b, label=f"{sim_b} upper")

    ax.set_xlabel(r"$\tau$")
    ax.set_ylabel(r"Quantile band value")

    if title is None:
        if mode == "fixed":
            title = rf"Bands (fixed): $\gamma_u$={float(gamma_u):.3f}, $\gamma_l$={float(gamma_l):.3f}"
        else:
            title = (
                rf"Bands (adaptive): "
                rf"{sim_a} $(\bar\gamma_u,\bar\gamma_l)=({out_a['gu']:.3f},{out_a['gl']:.3f})$, "
                rf"{sim_b} $(\bar\gamma_u,\bar\gamma_l)=({out_b['gu']:.3f},{out_b['gl']:.3f})$"
            )
    ax.set_title(title)

    if show_legend:
        ax.legend(fontsize=9, ncol=2, framealpha=0.3)

    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    plt.show()

    return {
        "tau": tau,
        sim_a: out_a,
        sim_b: out_b,
        "mode": mode,
    }



### Paper Figure 7 — Confidence Bands for GPT-4o vs Llama 3.3
This block reproduces the two-panel confidence-band comparison shown in Figure 7 of the paper.
Archived PNG: `figures/worldvalue_band_panels_gpt4o_vs_llama_n1000_n5000.png`.


In [ ]:
# ============================================================
# 2x1 vertical band plots: TWO simulators per panel
#   - top: n_list[0]
#   - bottom: n_list[1]
#   - in each panel: overlay bands for (sim_a, sim_b) with different colors
#   - indexing: alpha_low = bar_gamma_l * tau, alpha_up = (1-bar_gamma_u) + bar_gamma_u * tau
# ============================================================

def plot_two_band_panels_two_sims_adaptive(
    *,
    full_human,
    simulators_dict,
    sim_a,
    sim_b,
    n_list=(1000, 5000),
    k=200,
    g_u=g_default_n13,
    g_l=g_default_n13,
    ci_family="bounded",
    loss_kind="sq",
    ci_kwargs=None,
    seed=0,
    color_map=None,          # dict sim->color
    colors=None,             # optional override (color_a, color_b)
    tau_grid=None,
    figsize=(8.2, 9.2),
    sharex=True,
    fill_alpha=0.10,
    legend=True,
):
    """
    Two vertical panels (one per n_target). Each panel overlays the two simulator bands.

    IMPORTANT: uses bar_gamma_u/bar_gamma_l from compute_plus_minus_df_adaptive_gamma
    (i.e., bar over qids) and applies the asymptotic index map in quantile-coordinate form:
        lower uses  τ ↦ V^-_m( bar_gamma_l * τ )
        upper uses  τ ↦ V^+_m( (1-bar_gamma_u) + bar_gamma_u * τ )
    """
    ci_kwargs = ci_kwargs or {}

    # ----- common tau grid
    if tau_grid is None:
        tau = np.linspace(1e-3, 1.0 - 1e-3, 235)
    else:
        tau = np.asarray(tau_grid, float)
        tau = np.clip(tau, 1e-6, 1.0 - 1e-6)

    # ----- colors
    if colors is not None:
        col_a, col_b = colors
    else:
        col_a = None if color_map is None else color_map.get(sim_a, None)
        col_b = None if color_map is None else color_map.get(sim_b, None)

    fig, axes = plt.subplots(
        nrows=len(n_list),
        ncols=1,
        figsize=figsize,
        sharex=sharex,
        constrained_layout=True,
    )
    if len(n_list) == 1:
        axes = [axes]

    outputs = {}

    for ax, n_human in zip(axes, n_list):
        # ----- compute adaptive plus/minus df for this n_human (for ALL simulators)
        df_pm, summ = compute_plus_minus_df_adaptive_gamma(
            full_human=full_human,
            simulators_dict=simulators_dict,
            n=int(n_human),
            k=int(k),
            g_u=g_u,
            g_l=g_l,
            ci_family=ci_family,
            loss_kind=loss_kind,
            ci_kwargs=ci_kwargs,
            seed=seed,
        )

        # bar gammas (over qids)
        gu_bar = float(summ["bar_gamma_u"])
        gl_bar = float(summ["bar_gamma_l"])

        # index maps
        alpha_low = np.clip(gl_bar * tau, 0.0, 1.0)
        alpha_up  = np.clip((1.0 - gu_bar) + gu_bar * tau, 0.0, 1.0)

        def _band_for(sim_name):
            sub = df_pm[df_pm["sim"] == sim_name]
            if sub.empty:
                raise ValueError(f"No rows for sim='{sim_name}' at n={n_human}.")
            x_plus  = sub["delta_plus"].to_numpy(float)
            x_minus = sub["delta_minus"].to_numpy(float)
            Vminus = empirical_quantile_curve(x_minus, alpha_low)
            Vplus  = empirical_quantile_curve(x_plus,  alpha_up)
            return Vminus, Vplus, sub

        Vminus_a, Vplus_a, sub_a = _band_for(sim_a)
        Vminus_b, Vplus_b, sub_b = _band_for(sim_b)

        # ----- draw bands (two fills, two pairs of lines)
        ax.fill_between(tau, Vminus_a, Vplus_a, alpha=fill_alpha, color=col_a)
        ax.plot(tau, Vminus_a, "--", lw=1.8, color=col_a, label=f"{sim_a} lower")
        ax.plot(tau, Vplus_a,  "-.", lw=1.8, color=col_a, label=f"{sim_a} upper")

        ax.fill_between(tau, Vminus_b, Vplus_b, alpha=fill_alpha, color=col_b)
        ax.plot(tau, Vminus_b, "--", lw=1.8, color=col_b, label=f"{sim_b} lower")
        ax.plot(tau, Vplus_b,  "-.", lw=1.8, color=col_b, label=f"{sim_b} upper")

        ax.set_ylabel("Value")
        ax.set_title(
            rf"n={int(n_human)}, k={int(k)}; "
            rf"$\bar\gamma_u={gu_bar:.3f}$, $\bar\gamma_l={gl_bar:.3f}$"
        )
        ax.grid(True, alpha=0.25)
        if legend:
            ax.legend(loc="best", fontsize=9, ncol=2, framealpha=0.3)

        outputs[int(n_human)] = {
            "df_pm": df_pm,
            "gamma_summary": summ,
            "tau": tau,
            "alpha_low": alpha_low,
            "alpha_up": alpha_up,
            sim_a: {"Vminus": Vminus_a, "Vplus": Vplus_a},
            sim_b: {"Vminus": Vminus_b, "Vplus": Vplus_b},
        }

    axes[-1].set_xlabel(r"$\tau$")
    fig.suptitle(f"Adaptive bands overlay: {sim_a} vs {sim_b}", y=1.01)
    outputs["_figure"] = fig
    return outputs


# ===================== RUN IT (example) =====================
outs2 = plot_two_band_panels_two_sims_adaptive(
    full_human=full,
    simulators_dict=simulators,
    sim_a="GPT4o",
    sim_b="Llama",
    n_list=(1000, 5000),
    k=K_MODEL,
    g_u=g_default_n13,
    g_l=g_default_n13,
    ci_family=CI_FAMILY,
    loss_kind=LOSS_KIND,
    ci_kwargs=CI_KWARGS,
    seed=SEED,
    color_map=(SIM_COLOR_MAP if "SIM_COLOR_MAP" in globals() else None),
    figsize=(8.2, 9.2),
)
band_fig_path = FIGURES_DIR / "worldvalue_band_panels_gpt4o_vs_llama_n1000_n5000.png"
outs2["_figure"].savefig(band_fig_path, dpi=200, bbox_inches="tight")
print(f"Saved confidence-band figure to: {band_fig_path}")


### Supplementary Figure — Beta Sensitivity for $\gamma_j = 1 - n_j^{-\beta}$

This block varies the exponent $\beta$ in the adaptive schedule $\gamma_j = 1 - n_j^{-\beta}$ using the ordered grid $\beta \in \{1/5, 1/4, 1/3, 1/2\}$ and saves the final figure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import zlib
from numpy.random import default_rng

def _stable_int_hash(x) -> int:
    """Deterministic 32-bit hash for qid keys (works for str/int/etc.)."""
    b = str(x).encode("utf-8")
    return int(zlib.adler32(b))  # deterministic across runs

def _gamma_1_minus_n_pow_beta(n_used: int, beta: float, eps: float = 1e-6) -> float:
    n = max(1, int(n_used))
    g = 1.0 - n ** (-float(beta))
    return float(np.clip(g, eps, 1.0 - eps))


def _format_beta_label(beta: float) -> str:
    common = {0.2: r"1/5", 0.25: r"1/4", 1.0/3.0: r"1/3", 0.5: r"1/2"}
    for val, label in common.items():
        if np.isclose(beta, val):
            return label
    return f"{beta:.3g}"

def plot_indexed_curves_over_betas_side_by_side_n(
    *,
    full_human,
    sim_answers,                 # one simulator: dict/Series map qid -> responses
    sim_label="SIM",
    n_list=(1000, 5000),         # panels side-by-side
    betas=(1/5, 1/4, 1/3, 1/2),   # curves within each panel
    k=200,
    ci_family="bounded",
    loss_kind="sq",
    ci_kwargs=None,
    seed=0,
    tau_grid=None,
    figsize=None,
    sharey=True,
    legend_loc="best",
):
    """
    Side-by-side panels for n in n_list. Within each panel, overlays curves for betas.
    No oracle.

    IMPORTANT: Uses theorem-consistent asymptotic index shift:
        alpha_shifted(tau) = (1 - bar_gamma) + bar_gamma * tau,
    where bar_gamma is the average of gamma_j across the qids used in that panel/beta.
    """
    ci_kwargs = ci_kwargs or {}

    # ---- tau grid
    if tau_grid is None:
        tau = np.linspace(1e-3, 1.0 - 1e-3, 235)
    else:
        tau = np.asarray(tau_grid, float)
        tau = np.clip(tau, 1e-6, 1.0 - 1e-6)

    # ---- normalize inputs
    full_dict = _as_dict_series(full_human)
    sim_dict  = _as_dict_series(sim_answers)

    # ---- align qids
    qids = sorted(set(full_dict).intersection(sim_dict))
    if not qids:
        raise ValueError("No overlapping qids between full_human and sim_answers.")

    # ---- precompute fixed model subsample per qid (so curves differ only due to gamma/beta/n)
    rng_model = default_rng(int(seed) + 12345)
    y_model_k = {}
    for q in qids:
        ym_full = _dropna_np(sim_dict[q])
        if ym_full.size == 0:
            continue
        k_use = int(min(int(k), int(ym_full.size)))
        idx_m = rng_model.choice(ym_full.size, size=k_use, replace=False)
        y_model_k[q] = ym_full[idx_m]

    qids_use = [q for q in qids if q in y_model_k]
    if not qids_use:
        raise ValueError("No usable qids after dropping missing model data.")

    # ---- figure
    n_panels = len(n_list)
    if figsize is None:
        figsize = (6.6 * n_panels, 5.2)

    fig, axes = plt.subplots(1, n_panels, figsize=figsize, sharey=sharey, constrained_layout=True)
    if n_panels == 1:
        axes = [axes]

    cmap = plt.get_cmap("tab10")
    diag = {"tau": tau, "panels": {}}

    # ---- loop over n panels
    for ax, n_target in zip(axes, list(n_list)):
        n_target = int(n_target)

        # precompute fixed human subsample per qid for this n_target
        yh_sub = {}
        n_used_map = {}
        rng_h = default_rng(int(seed) + 10_000 + n_target)

        for q in qids_use:
            yh_full = _dropna_np(full_dict[q])
            if yh_full.size == 0:
                continue
            n_use = int(min(n_target, int(yh_full.size)))
            if n_use <= 0:
                continue
            idx_h = rng_h.choice(yh_full.size, size=n_use, replace=False)
            yh_sub[q] = yh_full[idx_h]
            n_used_map[q] = n_use

        qids_panel = [q for q in qids_use if q in yh_sub and len(yh_sub[q]) > 0]
        if not qids_panel:
            raise ValueError(f"No usable qids for n_target={n_target} after human subsampling.")

        panel_info = {"n_target": n_target, "betas": {}}

        # plot each beta curve
        for i, beta in enumerate(list(betas)):
            beta = float(beta)

            deltas = []
            gammas = []

            for q in qids_panel:
                n_use = n_used_map[q]
                gamma_j = _gamma_1_minus_n_pow_beta(n_use, beta, eps=1e-6)

                # deterministic per-(qid,n,beta) RNG so the CI randomness is comparable/run-stable
                h = _stable_int_hash(q)
                rng_local = default_rng((int(seed) * 1_000_003 + 97 * n_target + int(1e6 * beta) + h) % (2**32 - 1))

                d, _ = compute_pseudo_delta(
                    y_human=yh_sub[q],
                    y_model=y_model_k[q],
                    k=int(len(y_model_k[q])),
                    gamma=float(gamma_j),
                    ci_family=ci_family,
                    loss_kind=loss_kind,
                    ci_kwargs=ci_kwargs,
                    rng=rng_local,
                )
                if np.isfinite(d):
                    deltas.append(float(d))
                    gammas.append(float(gamma_j))

            deltas = np.asarray(deltas, float)
            gammas = np.asarray(gammas, float)

            if deltas.size == 0:
                continue

            bar_gamma = float(np.mean(gammas))
            alpha_shifted = np.clip((1.0 - bar_gamma) + bar_gamma * tau, 0.0, 1.0)

            curve = empirical_quantile_curve(deltas, alpha_shifted)

            color = cmap.colors[i % 10]
            beta_label = _format_beta_label(beta)
            label = rf"$\beta={beta_label}$ ($\bar\gamma={bar_gamma:.3f}$, m={len(deltas)})"
            ax.plot(tau, curve, lw=2.6, color=color, label=label)

            panel_info["betas"][beta] = {
                "m_eff": int(len(deltas)),
                "bar_gamma": bar_gamma,
                "alpha_shifted": alpha_shifted,
                "curve": curve,
            }

        ax.set_title(rf"{sim_label}: $n={n_target}$, $k={int(k)}$")
        ax.set_xlabel(r"$\tau$")
        ax.grid(True, alpha=0.25)
        ax.legend(loc=legend_loc, fontsize=9, framealpha=0.3)

        diag["panels"][n_target] = panel_info

    axes[0].set_ylabel(r"$\widehat V_m\!\left((1-\bar\gamma)+\bar\gamma\,\tau\right)$")
    return diag


In [ ]:
diag = plot_indexed_curves_over_betas_side_by_side_n(
    full_human=full,
    sim_answers=gpt4o,          # your simulator dict/series for GPT4o
    sim_label="GPT4o",
    n_list=(50, 200),        # side-by-side panels
    betas=(1/5, 1/4, 1/3, 1/2),  # curves in each panel
    k=K_MODEL,
    ci_family=CI_FAMILY,
    loss_kind=LOSS_KIND,
    ci_kwargs=CI_KWARGS,
    seed=SEED,
)


_beta_fig_path = FIGURES_DIR / 'worldvalue_beta_sensitivity_gpt4o_n50_n200.png'
_beta_fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.gcf().savefig(_beta_fig_path, dpi=200, bbox_inches="tight")
print(f"Saved beta-sensitivity figure to: {_beta_fig_path}")
